In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['CRYPTOGRAPHY_OPENSSL_NO_LEGACY'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

import datasets
from datasets import load_dataset, Dataset, DatasetDict, Dataset
from tqdm.auto import tqdm
import json

datasets.disable_caching()

/mnt/nas/home/genggui/miniconda3/envs/megatron-next/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from megatron.energon import get_train_dataset, WorkerConfig, get_loader, DefaultTaskEncoder, Cooker, basic_sample_keys
from megatron.energon import DefaultTaskEncoder, TextSample, Sample
import gzip
import pickle
import webdataset as wds
import json
import dataclasses
import torch

from tqdm.auto import tqdm
from typing import List

from qwen3_base_energon_helpers import MyTaskEncoder, print_error_handler

In [3]:
tokenizer_path="/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v19_2_122b_a10b_next_gemini_bf16/tokenizer_v19_gemini"

In [4]:
worker_config = WorkerConfig(
    rank=0,
    world_size=1,
    seed_offset=123456789,
    num_workers=1,
)

train_ds = get_train_dataset(
    '/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v19_2_122b_a10b_next_gemini_bf16/dataset.yaml',
    split_part="train",
    task_encoder=MyTaskEncoder(
        tokenizer_path=tokenizer_path,
        seq_length=66536,
        sensitive_words_path=None,
    ),
    batch_size=1,
    packing_buffer_size=8192,
    shuffle_buffer_size=None,
    max_samples_per_sequence=None,
    worker_config=worker_config,
)

train_dataloader = get_loader(train_ds, worker_config=worker_config, watchdog_timeout_seconds=240)

rank=0, worker=0: sample_range=[0, 98994] in 1 slices, sum(count)=98994: indexes=[0, 98994] slices=[train-chunk-0.tar[0(start), 98994(end)]]
rank=0, worker=0: sample_range=[0, 33580] in 1 slices, sum(count)=33580: indexes=[0, 33580] slices=[train-chunk-0.tar[0(start), 33580(end)]]
rank=0, worker=0: sample_range=[0, 70519] in 1 slices, sum(count)=70519: indexes=[0, 70519] slices=[train-chunk-0.tar[0(start), 70519(end)]]
rank=0, worker=0: sample_range=[0, 99256] in 1 slices, sum(count)=99256: indexes=[0, 99256] slices=[train-chunk-0.tar[0(start), 99256(end)]]
rank=0, worker=0: sample_range=[0, 14160] in 1 slices, sum(count)=14160: indexes=[0, 14160] slices=[train-chunk-0.tar[0(start), 14160(end)]]
rank=0, worker=0: sample_range=[0, 5231] in 1 slices, sum(count)=5231: indexes=[0, 5231] slices=[train-chunk-0.tar[0(start), 5231(end)]]
rank=0, worker=0: sample_range=[0, 1453] in 1 slices, sum(count)=1453: indexes=[0, 1453] slices=[train-chunk-0.tar[0(start), 1453(end)]]
rank=0, worker=0: sam

/mnt/nas/home/genggui/miniconda3/envs/megatron-next/lib/python3.12/site-packages/megatron/energon/loader.py:108: FutureWarning: Passing a worker_config to get_loader() is deprecated and will have no effect.
  warn_deprecated(


In [5]:
dd = iter(train_dataloader)

Filling reading buffer: 100%|██████████| 8192/8192 [01:13<00:00, 111.32it/s]
Traceback (most recent call last):
  File "/mnt/nas/home/genggui/miniconda3/envs/megatron-next/lib/python3.12/site-packages/megatron/energon/wrappers/map_dataset.py", line 136, in __iter__
    for idx, (sample_idx, inner_sample) in enumerate(
                                           ^^^^^^^^^^
  File "/mnt/nas/home/genggui/miniconda3/envs/megatron-next/lib/python3.12/site-packages/megatron/energon/wrappers/base.py", line 167, in iter_ctx
    x = next(it)
        ^^^^^^^^
  File "/mnt/nas/home/genggui/code/Megatron-Next/Pai-Megatron-Patch/examples/qwen35/qwen3_base_energon_helpers.py", line 247, in encode_sample
    yield from self.encode_text_sft_iter(sample, is_reasoning=True, only_use_last_message=False)
  File "/mnt/nas/home/genggui/code/Megatron-Next/Pai-Megatron-Patch/examples/qwen35/qwen3_base_energon_helpers.py", line 330, in encode_text_sft_iter
    build_text = self.tokenizer.apply_chat_template(
  

----------
SFTSample(__key__='train-chunk-0.tar/MSAgent-Bench-en-think-train-35', __restore_key__=('BlendDataset', 121, 'MapDataset', 35, 'MapDataset', 35, 'Webdataset', 35), __subflavors__={'data': 'data'}, __s...n Chinese.', 'role': 'assistant', 'tool_calls': [{'function': {'arguments': '{"movie_name":"肖申克的救赎"}', 'name': 'movie_info'}, 'id': 'call_NIGRz7Eadt7fnz5QR9qQsn8q', 'index': 0, 'type': 'function'}]}])
----------
MapDataset <bound method MyTaskEncoder.encode_sample of <qwen3_base_energon_helpers.MyTaskEncoder object at 0x7fb1bf1193d0>> failed 1/100 times in a row.


In [ ]:
import json

count_map = {}

with open("/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v19_2_122b_a10b_next_gemini_bf16/qua/pulse_v19_122b_a10b_gemini_train.jsonl", "w", encoding="utf8") as f:
    for _ in tqdm(range(2048)):
        item = next(dd)
        
        kk = item['__key__'][0].split("/")[-1].split("-train-")[0]
        if kk not in count_map:
            count_map[kk] = 0
        
        count_map[kk] += 1
              
        item = {
            "input_ids": item['input_ids'].tolist()[0],
            "labels": item['labels'].tolist()[0],
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        

            
        

In [ ]:
count_list = sorted(count_map.items(), key=lambda x:-x[1])
[(k,v/2048)for k,v in count_list]

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

In [ ]:
tokenizer("<|im_start|>assistant\n<ggthink>\n\n</ggthink>\n\n").input_ids 

In [ ]:
tokenizer("<|im_start|>assistant\n<ggthink>\n").input_ids

In [ ]:
tokenizer('<|im_end|>').input_ids

In [6]:
for _ in tqdm(range(2048)):
    item = next(dd)

100%|██████████| 2048/2048 [09:10<00:00,  3.72it/s]


In [9]:
item = next(dd)
# print(item['__key__'][0].split("/")[-1].split("-train-")[0])
print(tokenizer.decode([item for item in item['input_ids'].tolist()[0] if item != -100]))

<|im_start|>user
Devise a function that takes in three lists as parameters. The first two lists consist of integers, floats, strings, complex numbers, boolean values, special characters, null values, undefined values, NaN values, Infinity values, negative Infinity values, and possibly nested lists, while the third list contains a series of operations represented as strings. These operations can be "merge", "common_elements", "unique_elements", "sort_in_descending", "element_difference", "length_of_list", "sum_of_elements", "average_of_elements", "max_element", "min_element", "flatten_nested_list", "remove_duplicates", "find_median", "find_mode", "find_range", "find_standard_deviation", "find_variance", "find_percentile", "find_quartile", "find_interquartile_range". The function should perform the operations on the first two lists in the sequence they appear in the third list. The result of each operation should be displayed in a descending order of elements. If an operation cannot be p

In [10]:
print(tokenizer.decode([item for item in item['labels'].tolist()[0] if item != -100]))

Okay, let's devise this function. Handling all these diverse requirements, especially the mix of data types, special values, nested structures, statistical operations with dependencies, and error conditions, requires careful planning.

First, here is the requested piece of **incorrect code** for reference. This snippet attempts a simplified version but contains several flaws:

```python
# --- INCORRECT CODE SNIPPET ---
# This code is provided as a reference with intentional errors.
# Do NOT use this code directly.

import math

def buggy_process_lists(list1, list2, operations):
    results = []
    if not operations:
        return list1, list2 # Incorrect: Should return lists, not tuple? Format?

    # Problem 1: Modifies original lists implicitly sometimes?
    current_list1 = list1
    current_list2 = list2

    for op in operations:
        try:
            if op == "merge":
                # Problem 2: Incorrect merge logic (element-wise multiplication?)
                # Also, do